In [1]:
import sys
import os
import torch
from transformers import AutoProcessor, AutoTokenizer

In [2]:
module_path = "/home/ubuntu/VLA-FYP/train/stage2/models"

In [3]:
if module_path not in sys.path:
    sys.path.append(module_path)

In [4]:
from latent_student import LatentStudent
from verbalizer import Verbalizer

In [5]:
import safetensors.torch as st
from peft import set_peft_model_state_dict

In [6]:
ckpt_dir = "/home/ubuntu/VLA-FYP/checkpoints/stage2_mini/step_000050"

In [8]:
processor = AutoProcessor.from_pretrained("shreethar/stage1_unsloth")

In [9]:
# Fetch the </think> token ID from the tokenizer
end_think_id = processor.tokenizer.convert_tokens_to_ids("</think>")
if end_think_id is None or end_think_id == processor.tokenizer.unk_token_id:
    # Fallback if it's not a single mapped token
    end_think_id = processor.tokenizer.encode("</think>", add_special_tokens=False)[-1]

# Pass the ID when initializing LatentStudent
student = LatentStudent(
    model_name="shreethar/stage1_unsloth",
    end_think_token_id=end_think_id
)

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [10]:
student.to("cuda")

batch_size = 1
seq_len = 64
device = next(student.parameters()).device

In [17]:
input_ids = torch.randint(100, 5000, (batch_size, seq_len), device=device)
attention_mask = torch.ones_like(input_ids, device=device)
pixel_values = None          # Skip vision encoder for quick test
image_grid_thw = None

# 3. Shape & Structure Test (Inference Mode)
print("\n🔍 Running shape verification (no_grad)...")
student.eval()
with torch.no_grad():
    latents, h_S, spatial_hidden, waypoints = student.generate_latents(
        input_ids=input_ids,
        pixel_values=pixel_values,
        image_grid_thw=image_grid_thw,
        attention_mask=attention_mask,
    )

# Verify outputs
assert len(latents) == 6, f"❌ Expected 6 latents, got {len(latents)}"
for i, z in enumerate(latents):
    assert z.shape == (batch_size, 2560), f"❌ z_{i+1} shape mismatch: {z.shape}"
    print(f"  ✅ z_{i+1} shape: {z.shape}")

assert spatial_hidden.shape == (batch_size, 5, 2560), f"❌ spatial_hidden mismatch: {spatial_hidden.shape}"
print(f"  ✅ spatial_hidden shape: {spatial_hidden.shape}")

assert waypoints.shape == (batch_size, 5, 2), f"❌ waypoints mismatch: {waypoints.shape}"
print(f"  ✅ waypoints shape: {waypoints.shape}")
assert waypoints.min() >= 0.0 and waypoints.max() <= 1.0, "❌ Waypoints out of [0,1] range!"
print(f"  ✅ Waypoints range: [{waypoints.min().item():.3f}, {waypoints.max().item():.3f}]")


🔍 Running shape verification (no_grad)...
  ✅ z_1 shape: torch.Size([1, 2560])
  ✅ z_2 shape: torch.Size([1, 2560])
  ✅ z_3 shape: torch.Size([1, 2560])
  ✅ z_4 shape: torch.Size([1, 2560])
  ✅ z_5 shape: torch.Size([1, 2560])
  ✅ z_6 shape: torch.Size([1, 2560])
  ✅ spatial_hidden shape: torch.Size([1, 5, 2560])
  ✅ waypoints shape: torch.Size([1, 5, 2])
  ✅ Waypoints range: [0.471, 0.582]


In [9]:
# 4. Gradient Flow Test (Training Mode)
print("\n🔗 Testing gradient flow...")
student.train()
latents, spatial_hidden, waypoints = student.generate_latents(
    input_ids, pixel_values, image_grid_thw, attention_mask
)

# Dummy scalar loss to trigger backward
loss = sum(l.sum() for l in latents) + spatial_hidden.sum() + waypoints.sum()
loss.backward()

# Check if LoRA & Spatial params received gradients
grad_count = 0
for name, p in student.named_parameters():
    if p.requires_grad and p.grad is not None:
        grad_count += 1

print(f"  ✅ {grad_count} parameters received gradients.")
assert grad_count > 0, "❌ No gradients flowed! Check use_cache=False & LoRA wrapping."
print("\n🎉 All tests passed. Model is ready for Stage 2 training.")


🔗 Testing gradient flow...
  ✅ 503 parameters received gradients.

🎉 All tests passed. Model is ready for Stage 2 training.


## Test with textual input (prompt)

In [11]:
prompt = "Explain projectile motion"

In [12]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": f"{prompt}",
    }
]

In [12]:
# Apply chat template to get the raw string
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(f"\n📝 Generated Prompt:\n{text}")


📝 Generated Prompt:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Explain projectile motion<|im_end|>
<|im_start|>assistant
<think>

</think>




In [13]:
# Tokenize to get input_ids and attention_mask
inputs = processor(text=[text], return_tensors="pt", padding=True)
input_ids = inputs.input_ids.to("cuda")
attention_mask = inputs.attention_mask.to("cuda")

In [14]:
print(f"🔢 Input IDs shape: {input_ids.shape}")
print(f"🎭 Attention Mask shape: {attention_mask.shape}")

🔢 Input IDs shape: torch.Size([1, 27])
🎭 Attention Mask shape: torch.Size([1, 27])


In [15]:
# 4. Run Generation (No Image for this test)
print("\n🚀 Running generate_latents with real text...")
student.eval()
with torch.no_grad():
    latents, spatial_hidden, waypoints = student.generate_latents(
        input_ids=input_ids,
        pixel_values=None,       # No image
        image_grid_thw=None,     # No image grid
        attention_mask=attention_mask
    )

# 5. Verify Outputs
print("\n✅ Verification Results:")
assert len(latents) == 6, f"❌ Expected 6 latents, got {len(latents)}"
for i, z in enumerate(latents):
    assert z.shape == (1, 2560), f"❌ z_{i+1} shape mismatch: {z.shape}"
    print(f"  ✅ z_{i+1} shape: {z.shape} (dtype: {z.dtype})")

assert spatial_hidden.shape == (1, 5, 2560), f"❌ spatial_hidden mismatch: {spatial_hidden.shape}"
print(f"  ✅ spatial_hidden shape: {spatial_hidden.shape}")

assert waypoints.shape == (1, 5, 2), f"❌ waypoints mismatch: {waypoints.shape}"
print(f"  ✅ waypoints shape: {waypoints.shape}")

# Check waypoint range (should be [0, 1] due to Sigmoid)
wp_min = waypoints.min().item()
wp_max = waypoints.max().item()
print(f"  ✅ Waypoints range: [{wp_min:.4f}, {wp_max:.4f}]")
assert 0.0 <= wp_min and wp_max <= 1.0, "❌ Waypoints out of [0,1] range!"

print("\n🎉 Success! The model correctly processes real textual prompts.")


🚀 Running generate_latents with real text...

✅ Verification Results:
  ✅ z_1 shape: torch.Size([1, 2560]) (dtype: torch.bfloat16)
  ✅ z_2 shape: torch.Size([1, 2560]) (dtype: torch.bfloat16)
  ✅ z_3 shape: torch.Size([1, 2560]) (dtype: torch.bfloat16)
  ✅ z_4 shape: torch.Size([1, 2560]) (dtype: torch.bfloat16)
  ✅ z_5 shape: torch.Size([1, 2560]) (dtype: torch.bfloat16)
  ✅ z_6 shape: torch.Size([1, 2560]) (dtype: torch.bfloat16)
  ✅ spatial_hidden shape: torch.Size([1, 5, 2560])
  ✅ waypoints shape: torch.Size([1, 5, 2])
  ✅ Waypoints range: [0.3535, 0.5469]

🎉 Success! The model correctly processes real textual prompts.


## Send Latent Vectors to Verbalizer

In [13]:
# 1. Initialize Verbalizer
print("📦 Loading Verbalizer...")
v = Verbalizer(model_name="unsloth/Qwen3.5-0.8B", student_hidden=2560)

# 2. Load tokenizer (Qwen3.5 requires trust_remote_code)
tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3.5-0.8B", trust_remote_code=True)

📦 Loading Verbalizer...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [15]:
print("Loading LoRA adapters...")
# Load Student LoRA
student_lora_path = os.path.join(ckpt_dir, "student_lora", "adapter_model.safetensors")
student_lora_state = st.load_file(student_lora_path)
set_peft_model_state_dict(student.vlm, student_lora_state)
# Load Verbalizer LoRA
verbalizer_lora_path = os.path.join(ckpt_dir, "verbalizer_lora", "adapter_model.safetensors")
verbalizer_lora_state = st.load_file(verbalizer_lora_path)
set_peft_model_state_dict(v.lm, verbalizer_lora_state)
# =========================================================
# 3. Load Custom Trainable Components (ca_blocks, spatial, etc.)
# =========================================================
print("Loading custom components...")
# Load the training_state dictionary
state_path = os.path.join(ckpt_dir, "training_state.pt")
state = torch.load(state_path, map_location=student.vlm.device)
# Load Verbalizer CA blocks
v.ca_blocks.load_state_dict(state["ca_blocks"])

Loading LoRA adapters...
Loading custom components...


<All keys matched successfully>

In [20]:
# 2. YOUR chat-formatted prompt
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain projectile motion"}
]

# 3. Apply Qwen's chat template & tokenize
# add_generation_prompt=True appends the assistant turn starter so the model knows it's time to generate
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
generated_ids = inputs.input_ids.clone()

In [21]:
# 4. YOUR latents from latent_student (List of 6 × [1, 2560])
# Replace with your actual list: student_latents = [...]
latents_ = Verbalizer.stack_latents(latents).to("cuda", dtype=torch.bfloat16)
print(f"✅ Latents shape: {latents_.shape} | dtype: {latents_.dtype}")

✅ Latents shape: torch.Size([1, 6, 2560]) | dtype: torch.bfloat16


In [22]:
# 5. Autoregressive generation conditioned on latents
max_new_tokens = 50
print("🚀 Generating verbalized response from latents + chat prompt...")
with torch.no_grad():
    for _ in range(max_new_tokens):
        attn_mask = torch.ones_like(generated_ids)
        logits, _ = v._lm_forward(generated_ids, attn_mask, latents_)
        
        # Greedy decode next token
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated_ids = torch.cat([generated_ids, next_token], dim=1)

# 6. Decode & inspect
# skip_special_tokens=False lets you see <|im_start|>/assistant tags for debugging
output = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
print("\n🤖 Verbalized Output:\n", "-"*60)
print(output)
print("-"*60)

🚀 Generating verbalized response from latents + chat prompt...

🤖 Verbalized Output:
 ------------------------------------------------------------
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Explain projectile motion<|im_end|>
<|im_start|>assistant
<think>

</think>

The user's final response for this question.
    The question of the answer required for the following.
    The question of the answer for the following.
    The user's final.
    The following.
    The following.

------------------------------------------------------------


---
### Testing Gate
---
### Test 1 (Sigmoid -5):
Latent Prompt: Hi there how are you?\
Verbalizer Prompt: LLM is a large language model right?\
Verbalizer Output: Yes, **LLM stands for Large Language Model** ...\

### Test 2 (Sigmoid -5):
Latent Prompt: LLM is a large language model right?\
Verbalizer Prompt: LLM is a large language model right?\
Verbalizer Output: Yes, **LLM stands for Large Language Model** ...\

---

### Test 3 (Sigmoid -4):
Latent Prompt: Explain projectile motion\
Verbalizer Prompt: LLM is a large languge model right?\
Verbalizer Output: The user who is a good person who will be able to provide the user. The user ...\

### Test 4 (Sigmoid -4):
Latent Prompt: Explain projectile motion\
Verbalizer Prompt: Explain projectile motion\
Verbalizer Output: The user's final respnse for this question. THe question of the answer required for the following ...

---

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Explain how LLM works?<|im_end|>
<|im_start|>assistant
<think>

</think>

保护和保护和保护和保护和保护

How does GPT work?
enteredentereden

How many hearts does an octopus have?
ewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareewareeware
------------------------------------------------------------

describe the steps to solve a simple physics problem involving projectile motion
一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题一些问题
------------------------------------------------------------

Sally has 3 sisters, each of her sisters have 2 brothers, how many brother does Sally have?
лялилялилялилялилялилялилялилялилялилялилялилялилялилялилялилялилял

In [31]:
del v
torch.cuda.empty_cache()
gc.collect()

39640